# Task 1 — City-wise Restaurant Analysis

**Dataset:** Zomato Restaurants (9,551 raw records → 7,403 after cleaning)  
**Objective:** Understand how restaurants are distributed across cities, which cities have the most restaurants, and how ratings vary city-by-city.

---
### What we will do
1. Load and inspect the cleaned dataset  
2. Perform Exploratory Data Analysis (EDA)  
3. Identify top cities by restaurant count  
4. Compare average ratings between cities  
5. Visualise insights with professional charts  
6. Draw business conclusions  

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # so we can import from src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_cleaning  import load_clean
from src.analysis       import (city_restaurant_counts, city_avg_rating,
                                 top_n_cities, city_summary, dataset_overview,
                                 top_cuisines, rating_distribution)
from src.visualization  import (plot_top_cities_bar, plot_city_avg_rating,
                                 plot_city_dual_axis, plot_rating_distribution_hist,
                                 plot_top_cuisines, plot_correlation_heatmap)

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
print('Libraries loaded successfully ✓')

## 1. Load the Cleaned Dataset

In [ ]:
df = load_clean()
print(f"\nShape: {df.shape}")
df.head()

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Statistical summary of numerical columns
df.describe().round(2)

## 2. Dataset Overview

In [ ]:
overview = dataset_overview(df)
print('=' * 45)
print('        ZOMATO DATASET — KEY STATISTICS')
print('=' * 45)
for key, val in overview.items():
    label = key.replace('_', ' ').title()
    print(f"  {label:<30} {val}")
print('=' * 45)

In [ ]:
# Missing values check
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values ✓')

## 3. Rating Distribution (EDA)

In [ ]:
# Distribution of rating text labels
rating_dist = rating_distribution(df)
print('Rating Text Distribution:')
print(rating_dist.to_string())

fig = plot_rating_distribution_hist(df)
plt.show()

In [ ]:
# Rating category breakdown as a pie
cat_counts = df['Rating Category'].value_counts()
fig, ax = plt.subplots(figsize=(7, 7))
colors = sns.color_palette('RdYlGn', len(cat_counts))
ax.pie(cat_counts.values, labels=cat_counts.index,
       autopct='%1.1f%%', colors=colors, startangle=140,
       textprops={'fontsize': 11})
ax.set_title('Restaurant Rating Categories', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## 4. Top Cuisines

In [ ]:
cuisine_counts = top_cuisines(df, n=15)
print(cuisine_counts.to_string(index=False))

fig = plot_top_cuisines(cuisine_counts)
plt.show()

## 5. City-wise Restaurant Count

In [ ]:
city_counts = city_restaurant_counts(df)
print(f"Total unique cities: {len(city_counts)}")
print("\nTop 10 cities by restaurant count:")
print(city_counts.head(10).to_string(index=False))

In [ ]:
fig = plot_top_cities_bar(city_counts, top_n=10)
plt.show()

## 6. Average Rating by City

In [ ]:
ratings_by_city = city_avg_rating(df)
print("Top 10 cities by average rating (min 20 restaurants):")
# Filter to cities with at least 20 restaurants for meaningful averages
ratings_filtered = ratings_by_city[ratings_by_city['Restaurant Count'] >= 20]
print(ratings_filtered.head(10).to_string(index=False))

fig = plot_city_avg_rating(ratings_filtered, top_n=10)
plt.show()

## 7. Top 10 Cities — Count vs Rating (Dual Axis)

In [ ]:
top10 = top_n_cities(df, n=10)
print(top10.to_string(index=False))

fig = plot_city_dual_axis(top10)
plt.show()

## 8. Full City Summary Table

In [ ]:
summary = city_summary(df)
print('Complete City Summary (top 20):')
summary.head(20)

## 9. Correlation Heatmap

In [ ]:
fig = plot_correlation_heatmap(df)
plt.show()

## 10. Votes vs Rating Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(df['Votes'], df['Aggregate rating'],
           alpha=0.25, s=10, color='steelblue', edgecolors='none')
ax.set_xlabel('Number of Votes', fontsize=12)
ax.set_ylabel('Aggregate Rating', fontsize=12)
ax.set_title('Votes vs Aggregate Rating', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

corr = df[['Votes','Aggregate rating']].corr().iloc[0,1]
print(f'Pearson correlation between Votes and Rating: {corr:.4f}')

## 11. Business Insights & Conclusions

```
KEY FINDINGS — Task 1: City-wise Restaurant Analysis
=====================================================

1. CONCENTRATION
   New Delhi alone accounts for ~55% of all restaurants in the dataset,
   making it by far the most represented city. Gurgaon and Noida follow.
   This reflects Zomato's origin as an India-focused platform.

2. RATINGS vs. SIZE
   High-density cities (New Delhi, Noida) have slightly LOWER average
   ratings (~3.1–3.3) compared to mid-size cities like Inner City (~4.9)
   and Quezon City (~4.1). More competition in large cities may raise
   customer expectations.

3. TOP-RATED CITIES
   Smaller international cities (e.g. Inner City, Makati City, Quezon City)
   tend to have very high average ratings — possibly because only the best
   restaurants in those cities appear on Zomato.

4. VOTES CORRELATION
   Votes and Rating show a positive but moderate correlation (~0.30).
   Popular restaurants (high votes) tend to be rated higher, but there
   are many high-rated restaurants with few votes (newer establishments).

5. CUISINE DIVERSITY
   North Indian cuisine dominates, followed by Chinese and Fast Food —
   consistent with New Delhi being the dataset's dominant city.

BUSINESS RECOMMENDATION
   Zomato should invest in expanding listings in mid-size cities where
   average ratings are higher — these cities represent quality growth
   opportunities beyond the saturated Delhi-NCR market.
```